Sprint 1 - Importando os Dados.

In [22]:

#Importando os Dados:
import pandas as pd
import numpy as np
import csv
from IPython.display import display

caminho = "dataset/Base Varejo.csv"

with open(caminho, "r", encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo, delimiter=";") # sem o delimiter a base aparece como uma única coluna. Isso indica que o CSV utiliza ";" como separador.
    dados = list(leitor)

varejo = pd.DataFrame(dados)

print("Informações da Base de Dados")
print("Quantidade de registros:", len(varejo))
print("Quantidade de colunas:", len(varejo.columns))

print("Colunas:")
print(varejo.columns.tolist())

print("Tipos de dados:")
print(varejo.dtypes)

print("Primeiras linhas:")
display(varejo.head())

Informações da Base de Dados
Quantidade de registros: 830000
Quantidade de colunas: 11
Colunas:
['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME', '']
Tipos de dados:
DATA         str
CO_ID        str
CL_ID        str
CL_GENERO    str
CL_EC        str
CL_FHL       str
CL_SEG       str
PR_ID        str
PR_CAT       str
PR_NOME      str
             str
dtype: object
Primeiras linhas:


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,


quando usamos o csv.DictReader, o shape ficou com 11 colunas porque o arquivo possui colunas extras sem nome no final do cabeçalho. Como o DictReader transforma cada linha em um dicionário, colunas com o mesmo nome vazio não são mantidas separadamente, ele junta tudo numa coluna " "  como podemos ver no tipo de dados após o PR_NOME. 

In [35]:
#para verificar o cabeçalho real: 
import csv

arquivo = "dataset/Base Varejo.csv"

with open(arquivo, mode="r", encoding="utf-8-sig", newline="") as csvfile:
    leitor_csv = csv.reader(csvfile, delimiter=";")
    cabecalho = next(leitor_csv)

print("Quantidade de colunas no cabeçalho bruto:")
print(len(cabecalho))

print("Cabeçalho bruto:")
print(cabecalho)

Quantidade de colunas no cabeçalho bruto:
14
Cabeçalho bruto:
['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME', '', '', '', '']


Como citado a cima, aqui aparecem as 4 colunas vazias, que se juntam no DictRead de forma automática transformando em apenas 1 coluna " " 

Sprint 2  - Transformação dos Dados.

In [36]:

# Limpando e padronizando as Strings
# aqui estamos criando uma lista chamada colunas_texto, que contem todas essas colunas dentro desta lista.
colunas_texto = [
    "CL_GENERO",
    "CL_EC",
    "CL_FHL",
    "PR_CAT",
    "PR_NOME"
]

for coluna in colunas_texto:
    varejo[coluna] = (
        varejo[coluna]
        .astype(str)                             # garante string mesmo se vier tipo misto
        .str.strip()                             # remove os espaços nas pontas
        .str.replace(r"\s+", " ", regex=True)    # remove os espaços internos duplicados ex: " João   Silva" -> "João Silva"
        .replace("NAN", pd.NA)                   # astype(str) transforma NaN em "nan" -> corrige aqui
    )

#Convertendo colunas numéricas inteiras
#aqui é o mesmo processo de criar a lista com as colunas desejadas dentro dela. 
colunas_num_inteiras = [
    "CO_ID",
    "CL_ID",
    "PR_ID"
]
#a função for percorre cada item de cada lista e retorna com a conversão de cada item de cada coluna, um de cada vez. 
for coluna in colunas_num_inteiras:
    varejo[coluna] = pd.to_numeric(
        varejo[coluna],
        errors="coerce"
    ).astype("Int64")

    n_invalidos = varejo[coluna].isna().sum()
    if n_invalidos > 0:
        print(f"[Aviso] {coluna}: {n_invalidos} valores não convertidos (viraram NA)")

#Convertendo a data para o padrão:

varejo["DATA"] = pd.to_datetime(
    varejo["DATA"],
    errors="coerce",
    dayfirst=True                      # transforma para o padrão brasileiro com os dias primeiro. ex dd/mm/aaaa
)
#aqui é um mecanismo para o sistema informar caso tenha uma data equivocada que não conseguiu ser convertida.
n_datas_invalidas = varejo["DATA"].isna().sum()
if n_datas_invalidas > 0:
    print(f"[Aviso] DATA: {n_datas_invalidas} valores não convertidos (viraram NaT)")

#Resultado das colunas depois das transformações
print("Resultados das colunas depois das transformações:")
print(varejo.dtypes)

Resultados das colunas depois das transformações:
DATA         datetime64[us]
CO_ID                 Int64
CL_ID                 Int64
CL_GENERO               str
CL_EC                   str
CL_FHL                  str
CL_SEG              float64
PR_ID                 Int64
PR_CAT                  str
PR_NOME                 str
                        str
dtype: object


Perceba que a coluna 11 que está vazia permanece. vamos remover ela no próximo Sprint.

Sprint 3 - Removendo Nulos e duplicatas.

In [70]:
#Vamos verificar os valores nulos com o isnull e depois somar todos os nulos com a função .sum()

print("Valores nulos em cada coluna:")
nulos = varejo.isnull().sum()    
print(nulos[nulos > 0])


#Conferindo as categorias vazias.
print("Categorias vazias:")

categorias_vazias = (
    varejo["PR_CAT"].isna() |
    (varejo["PR_CAT"].str.strip() == "")
).sum()

print("Categorias vazias:", categorias_vazias)

#Tratando as categorias vazias.

varejo["PR_CAT"] = varejo["PR_CAT"].fillna("SEM CATEGORIA")
varejo["PR_CAT"] = varejo["PR_CAT"].replace("", "SEM CATEGORIA")


#Tratando as categorias completamente vazias, onde a coluna inteira está vazia.
colunas_vazias = varejo.columns[varejo.isna().all()]

print("A Coluna inteira está vazia:")
print(colunas_vazias.tolist())


#Remove as colunas que possuem todos os valores nulos
varejo = varejo.dropna(axis=1, how="all")


#vamos conferir as duplicatas agora. 

duplicatas = varejo.duplicated().sum()
print("Duplicatas encontradas:", duplicatas)

#Agora vamos remover as duplicatas.
varejo = varejo.drop_duplicates()

print("Duplicatas após a limpeza:",
      varejo.duplicated().sum())


#Conferindo se existe mais algum nulo após a limpeza.

print("Nulos após a limpeza:")
nulos_final = varejo.isnull().sum()

print(nulos_final[nulos_final > 0])



Valores nulos em cada coluna:
Unnamed: 10    830000
Unnamed: 11    830000
Unnamed: 12    830000
Unnamed: 13    830000
dtype: int64
Categorias vazias:
Categorias vazias: 0
A Coluna inteira está vazia:
['Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']
Duplicatas encontradas: 96553
Duplicatas após a limpeza: 0
Nulos após a limpeza:
Series([], dtype: int64)


Sprint 4 (Estatística Descritiva): Aplicação das funções estatísticas para coletar parâmetros da coluna de Número de filhos do cliente.


In [68]:
#Convertendo  a coluna com número de filhos:

#aqui estou acessando a coluna CL_FHL que fica dentro do dataset varejo e pedindo para transformar os valores dentro desta coluna em números
varejo["CL_FHL"] = pd.to_numeric(
    varejo["CL_FHL"],
    errors="coerce"
)


#Estatisticas:

print("Número de Filhos:")

print("Contagem:", varejo["CL_FHL"].count())

print("Média:", varejo["CL_FHL"].mean())

print("Mediana:", varejo["CL_FHL"].median())

print("Desvio padrão:", varejo["CL_FHL"].std())

print("Máximo:", varejo["CL_FHL"].max())

print("Mínimo:", varejo["CL_FHL"].min())


#calculando a moda:

print("Moda:")

print(varejo["CL_FHL"].mode().tolist())


#Calculando os Quartis:

print("Quartis:")

Q1 = varejo["CL_FHL"].quantile(0.25)
Q3 = varejo["CL_FHL"].quantile(0.75)

print("Q1 (25%):", varejo["CL_FHL"].quantile(0.25))

print("Q2 (50%):", varejo["CL_FHL"].quantile(0.50))

print("Q3 (75%):", varejo["CL_FHL"].quantile(0.75))

#Identificando possiveis outliers:

IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print("Resultado analise outliers:")

print("IQR:", IQR)

print("Limite inferior:", limite_inferior)

print("Limite superior:", limite_superior)


# Identifica os registros na coluna CL_FHL dentro da variavel varejo possiveis outlier pelo critério estatistico( se for superior ou infeior)
# ao limite determinado nas formulas acima(limite_inferior e limite_superior)
outliers = varejo[
    (varejo["CL_FHL"] < limite_inferior) |
    (varejo["CL_FHL"] > limite_superior)
]

print("Quantidade de possíveis outliers:", len(outliers))

#Resumo das estatísticas

print("Resumo das estatisticas:")

print(varejo["CL_FHL"].describe())

Número de Filhos:
Contagem: 830000
Média: 1.1465397590361446
Mediana: 0.0
Desvio padrão: 1.4169595270011563
Máximo: 4
Mínimo: 0
Moda:
[0]
Quartis:
Q1 (25%): 0.0
Q2 (50%): 0.0
Q3 (75%): 2.0
Resultado analise outliers:
IQR: 2.0
Limite inferior: -3.0
Limite superior: 5.0
Quantidade de possíveis outliers: 0
Resumo das estatisticas:
count    830000.00000
mean          1.14654
std           1.41696
min           0.00000
25%           0.00000
50%           0.00000
75%           2.00000
max           4.00000
Name: CL_FHL, dtype: float64


5 Sprint -  Relatório final da base limpa

In [76]:


print("Resumo da database limpa:")

registros_limpos = len(varejo)
colunas_limpas = len(varejo.columns)

# Como as linhas removidas foram as duplicatas:
registros_antes_duplicatas = registros_limpos + duplicatas

print(
    f"Registros antes da remoção das duplicatas: "
    f"{registros_antes_duplicatas:,}"
)

print(
    f"Registros após a limpeza: "
    f"{registros_limpos:,}"
)

print(
    f"Registros duplicados removidos: "
    f"{duplicatas:,}"
)

print(
    f"Quantidade de colunas após a limpeza: "
    f"{colunas_limpas}"
)



#Dados depois da limpeza

print("Qualidade dos dados após a limpeza:")


nulos_restantes = varejo.isnull().sum().sum()
duplicatas_restantes = varejo.duplicated().sum()

print(
    f"Total de valores nulos restantes: "
    f"{nulos_restantes}"
)

print(
    f"Duplicatas restantes: "
    f"{duplicatas_restantes}"
)

print(
    f"Datas inválidas identificadas: "
    f"{n_datas_invalidas}"
)


# Analisando o periodo.

print("Período das análises")


data_inicial = varejo["DATA"].min()
data_final = varejo["DATA"].max()

quantidade_datas = varejo["DATA"].nunique()

print(
    "Data inicial:",
    data_inicial.strftime("%d/%m/%Y") #escreve a data en dia/mês/ano
)

print(
    "Data final:",
    data_final.strftime("%d/%m/%Y")
)

print(
    "Quantidade de datas diferentes:",
    quantidade_datas
)



#Estatistica de número de filhos


print("Estatisticas dos filhos:")

#criando a variavel filhos, onde contem apenas a coluna "CL_FHL" onde se encontram as informações sobre a quantidade de filhos.
filhos = varejo["CL_FHL"]
# aqui cada função tras um resultado esperado
contagem_filhos = filhos.count() #conta quantos valores válidos existem na coluna.
media_filhos = filhos.mean() #calcula a média do número de filhos.
mediana_filhos = filhos.median()  #calcula a mediana 
desvio_filhos = filhos.std() #calcula quanto os valores variam em relação a média
moda_filhos = filhos.mode().iloc[0] #encontra o valor que mais se repete.
minimo_filhos = filhos.min() #encontra o menor número de filhos
maximo_filhos = filhos.max() #encontra o maior número de filhos.

q1_filhos = filhos.quantile(0.25)
q2_filhos = filhos.quantile(0.50)
q3_filhos = filhos.quantile(0.75)


#Criando uma tabela com todas as estatísticas exigidas
estatisticas_filhos = pd.DataFrame({
    "ESTATÍSTICA": [
        "Contagem",
        "Média",
        "Mediana",
        "Desvio padrão",
        "Moda",
        "Mínimo",
        "Q1 (25%)",
        "Q2 (50%)",
        "Q3 (75%)",
        "Máximo"
    ],

    "VALOR": [
        contagem_filhos,
        round(media_filhos, 2),
        mediana_filhos,
        round(desvio_filhos, 2),
        moda_filhos,
        minimo_filhos,
        q1_filhos,
        q2_filhos,
        q3_filhos,
        maximo_filhos
    ]
})

display(estatisticas_filhos)



#Agrupamento por gênero


print("Agrupando por gênero")


agrupamento_genero = (
    varejo
    .groupby("CL_GENERO")
    .size()
    .reset_index(name="QUANTIDADE")
    .sort_values(
        "QUANTIDADE",
        ascending=False
    )
)

display(agrupamento_genero)

genero_maior = agrupamento_genero.iloc[0]["CL_GENERO"]
quantidade_genero = agrupamento_genero.iloc[0]["QUANTIDADE"]

print(
    f"Gênero com maior quantidade de registros: "
    f"{genero_maior}"
)

print(
    f"Quantidade: "
    f"{quantidade_genero:,}"
)


#Agrupando por categoria

print("Agrupando por categoria")

agrupamento_categoria = (
    varejo
    .groupby("PR_CAT")
    .size()
    .reset_index(name="QUANTIDADE")
    .sort_values(
        "QUANTIDADE",
        ascending=False
    )
)

display(agrupamento_categoria)

categoria_maior = agrupamento_categoria.iloc[0]["PR_CAT"]
quantidade_categoria = agrupamento_categoria.iloc[0]["QUANTIDADE"]

print(
    f"Categoria com maior quantidade de registros: "
    f"{categoria_maior}"
)

print(
    f"Quantidade: "
    f"{quantidade_categoria:,}"
)



#Agrupando por genero e categoria

print("Agrupamento por gênero e categoria")


agrupamento_genero_categoria = (
    varejo
    .groupby(
        ["CL_GENERO", "PR_CAT"]
    )
    .size()
    .reset_index(name="QUANTIDADE")
    .sort_values(
        "QUANTIDADE",
        ascending=False
    )
)

# Mostra somente os 10 maiores resultados, para evitar que o jupyter tranque a saída (aconteceu várias vezes durante a elaboração)
display(
    agrupamento_genero_categoria.head(10)
)

maior_combinacao = agrupamento_genero_categoria.iloc[0]

print(
    "Combinação com maior quantidade de registros:"
)

print(
    f"Gênero: {maior_combinacao['CL_GENERO']}"
)

print(
    f"Categoria: {maior_combinacao['PR_CAT']}"
)

print(
    f"Quantidade: {maior_combinacao['QUANTIDADE']:,}"
)



#validando o identificador de compras


print("Validando o identificador de compras - CO_ID")


compras_unicas = varejo["CO_ID"].nunique()

registros_por_compra = (
    varejo
    .groupby("CO_ID")
    .size()
)

media_registros_compra = registros_por_compra.mean()
menor_registros_compra = registros_por_compra.min()
maior_registros_compra = registros_por_compra.max()

print(
    f"Quantidade de compras únicas: "
    f"{compras_unicas:,}"
)

print(
    f"Média de registros por compra: "
    f"{media_registros_compra:.2f}"
)

print(
    f"Menor quantidade de registros em uma compra: "
    f"{menor_registros_compra}"
)

print(
    f"Maior quantidade de registros em uma compra: "
    f"{maior_registros_compra}"
)




#Os principais insights encontrados foram:


print("Principais Insights encontrados foram:")


print(
    f"Após a limpeza, a base ficou com "
    f"{registros_limpos:,} registros e "
    f"{colunas_limpas} colunas."
)

print(
    f"Foram removidos {duplicatas:,} "
    f"registros duplicados."
)

print(
    f"O número médio de filhos dos clientes foi de "
    f"{media_filhos:.2f}, com mediana de "
    f"{mediana_filhos} e moda de {moda_filhos}."
)

print(
    f"O gênero com maior quantidade de registros foi "
    f"{genero_maior}, com {quantidade_genero:,} registros."
)

print(
    f"A categoria com maior quantidade de registros foi "
    f"{categoria_maior}, com "
    f"{quantidade_categoria:,} registros."
)

print(
    f"A combinação de gênero e categoria mais frequente foi "
    f"{maior_combinacao['CL_GENERO']} + "
    f"{maior_combinacao['PR_CAT']}, com "
    f"{maior_combinacao['QUANTIDADE']:,} registros."
)



#conclusão sobre os resultados.


print("Conclusão: após o tratamento dos dados, a base ficou sem duplicatas e sem valores nulos relevantes.")
print("As estatisticas descritivas e os agrupamentos permitiram identificar caracteristicas dos clientes e padrões")
print("relacionados as categorias ed produtos")


print("Final do relatório.")


Resumo da database limpa:
Registros antes da remoção das duplicatas: 830,000
Registros após a limpeza: 733,447
Registros duplicados removidos: 96,553
Quantidade de colunas após a limpeza: 10
Qualidade dos dados após a limpeza:
Total de valores nulos restantes: 0
Duplicatas restantes: 0
Datas inválidas identificadas: 0
Período das análises
Data inicial: 04/01/2019
Data final: 08/12/2022
Quantidade de datas diferentes: 333
Estatisticas dos filhos:


,ESTATÍSTICA,VALOR
0,Contagem,733447.00
1,Média,1.15
2,Mediana,0.00
3,Desvio padrão,1.42
4,Moda,0.00
5,Mínimo,0.00
6,Q1 (25%),0.00
7,Q2 (50%),0.00
8,Q3 (75%),2.00
9,Máximo,4.00


Agrupando por gênero


,CL_GENERO,QUANTIDADE
0,F,382427
1,M,351020


Gênero com maior quantidade de registros: F
Quantidade: 382,427
Agrupando por categoria


,PR_CAT,QUANTIDADE
2,ALIMENTOS,384197
4,HIGIENE,137702
5,LIMPEZA,128632
3,BEBIDAS,38264
6,PET,28553
1,ACESSORIOS,12871
0,#N/D,3228


Categoria com maior quantidade de registros: ALIMENTOS
Quantidade: 384,197
Agrupamento por gênero e categoria


,CL_GENERO,PR_CAT,QUANTIDADE
2,F,ALIMENTOS,200274
9,M,ALIMENTOS,183923
4,F,HIGIENE,71721
5,F,LIMPEZA,67328
11,M,HIGIENE,65981
12,M,LIMPEZA,61304
3,F,BEBIDAS,19764
10,M,BEBIDAS,18500
6,F,PET,14809
13,M,PET,13744


Combinação com maior quantidade de registros:
Gênero: F
Categoria: ALIMENTOS
Quantidade: 200,274
Validando o identificador de compras - CO_ID
Quantidade de compras únicas: 18,471
Média de registros por compra: 39.71
Menor quantidade de registros em uma compra: 1
Maior quantidade de registros em uma compra: 81
Principais Insights encontrados foram:
Após a limpeza, a base ficou com 733,447 registros e 10 colunas.
Foram removidos 96,553 registros duplicados.
O número médio de filhos dos clientes foi de 1.15, com mediana de 0.0 e moda de 0.
O gênero com maior quantidade de registros foi F, com 382,427 registros.
A categoria com maior quantidade de registros foi ALIMENTOS, com 384,197 registros.
A combinação de gênero e categoria mais frequente foi F + ALIMENTOS, com 200,274 registros.
Conclusão: após o tratamento dos dados, a base ficou sem duplicatas e sem valores nulos relevantes.
As estatisticas descritivas e os agrupamentos permitiram identificar caracteristicas dos clientes e padrões
